Use SemEval 2020 Task 11 evaluation critique to measure how well our benchmark performs.

In [1]:
import pandas as pd
import numpy as np
import os
import torch
import zipfile
import shutil
import gdown
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification

In [2]:
#Identify base directory to ensure portability
BASE_DIR = Path.cwd().resolve().parent
MODELS_DIR = BASE_DIR / "models"
DATA_PATH = BASE_DIR / "data" / "processed" / "semeval_tc_cleaned.csv"
SI_DIR = MODELS_DIR / "semeval_roberta_scanner"
TC_DIR = MODELS_DIR / "semeval_roberta_classifier"

SI_MODEL_PATH = f"{os.fspath(SI_DIR.absolute())}"
TC_MODEL_PATH = f"{os.fspath(TC_DIR.absolute())}"


device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [3]:
test_df = pd.read_csv(DATA_PATH)
#Drop duplicate articles
articles_df = test_df[['article_id','text_content']].drop_duplicates(subset=['article_id'])

exclude = ['article_id', 'text_content', 'span_text', 'sentiment', 'punct_count', 'lexical_diversity', 'start', 'end']
label_cols = [c for c in test_df.columns if c not in exclude]
print(f"Loaded test dataset with {len(test_df)} spans.")

Loaded test dataset with 7587 spans.


In [4]:
#Run training notebooks if models are missing
REQUIRED_FILES = ["config.json", "model.safetensors"]

def model_exists(path):
    path = Path(path)
    has_weights = any(path.glob("*.bin")) or any(path.glob("*.safetensors"))
    return has_weights

if not model_exists(SI_MODEL_PATH):
    print("SI Model missing. Running training notebook...")
if not model_exists(TC_MODEL_PATH):
    print("TC Model missing. Running training notebook...")
    %run 4.2-fp-semeval-tc-modeling.ipynb

In [5]:
#Official SemEval 2020 Task 11 SI Evaluation Equation
def get_si_metrics(predicted_spans, gold_spans):
    """
    Implements SemEval-2020 Task 11 character-level overlap.
    Eq 1: P = 1/|S| * sum(|s ∩ t| / |s|)
    Eq 2: R = 1/|T| * sum(|s ∩ t| / |t|)
    """
    if not predicted_spans: return 0.0, 0.0, 0.0
    if not gold_spans: return 0.0, 0.0, 0.0

    def merge_spans(spans):
        if not spans: return []
        sorted_spans = sorted(spans)
        merged = [list(sorted_spans[0])]
        for curr in sorted_spans[1:]:
            prev = merged[-1]
            if curr[0] <= prev[1]:
                prev[1] = max(prev[1], curr[1])
            else:
                merged.append(list(curr))
        return [tuple(m) for m in merged]

    #Pre-merge overlapping spans as required by SemEval
    S = merge_spans(predicted_spans)
    T = merge_spans(gold_spans)

    #Precision calculation
    prec_sum = 0
    for s in S:
        overlap = 0
        for t in T:
            intersect = max(0, min(s[1], t[1]) - max(s[0], t[0]))
            overlap += intersect
        prec_sum += (overlap / (s[1] - s[0]))

    #Recall calculation
    rec_sum = 0
    for t in T:
        overlap = 0
        for s in S:
            intersect = max(0, min(s[1], t[1]) - max(s[0], t[0]))
            overlap += intersect
        rec_sum += (overlap / (t[1] - t[0]))

    precision = prec_sum / len(S)
    recall = rec_sum / len(T)
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1

In [6]:
#Load SI Model (RoBERTa token-classifier for span detection)
print(f"Loading SI Model from: {SI_MODEL_PATH}...")
si_tokenizer = AutoTokenizer.from_pretrained(SI_MODEL_PATH)
si_model = AutoModelForTokenClassification.from_pretrained(SI_MODEL_PATH, local_files_only=True).to(device)

Loading SI Model from: /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/semeval_roberta_scanner...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [7]:
#Load TC Model (Technique Classification)
tc_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
tc_model = AutoModelForSequenceClassification.from_pretrained(TC_MODEL_PATH).to(device)
tc_model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [8]:
def run_pipeline(text):
    #Span identification step
    inputs = si_tokenizer(text, return_tensors="pt", truncation=True, padding=True, return_offsets_mapping=True).to(device)
    offsets = inputs.pop("offset_mapping")[0]

    with torch.no_grad():
        logits = si_model(**inputs).logits
        predictions = torch.argmax(logits, dim=-1)[0]

    predicted_spans = []
    current_span = None
    for i, pred in enumerate(predictions):
        label = pred.item()
        start, end = offsets[i]
        if start == end: continue

        if label == 1:
            if current_span is None:
                current_span = [start.item(), end.item()]
            else:
                current_span[1] = end.item()
        else:
            if current_span:
                predicted_spans.append(tuple(current_span))
                current_span = None
    if current_span: predicted_spans.append(tuple(current_span))

    #Technique classification step
    final_results = []
    for span in predicted_spans:
        span_text = text[span[0]:span[1]]
        tc_inputs = tc_tokenizer(span_text, return_tensors="pt", truncation=True, padding=True).to(device)
        with torch.no_grad():
            tc_logits = tc_model(**tc_inputs).logits
            pred_class = torch.argmax(tc_logits, dim=1).item()

        #Get technique name (e.g., 'Slogans') from the model config
        tech_name = tc_model.config.id2label[pred_class]
        final_results.append({"span": span, "technique": tech_name})

    return final_results

In [11]:
all_si_scores = []
all_pipeline_scores = []

print(f"Evaluating pipeline on {test_df['article_id'].nunique()} articles...")

for _, row in articles_df.iterrows():
    article_id = row['article_id']
    full_text = row['text_content']

    #Get all human-labeled spans for this article
    gold_rows = test_df[test_df['article_id'] == article_id]
    gold_data = []
    for _, g_row in gold_rows.iterrows():
        #Identify the technique (find which column is 1)
        techs = [col for col in label_cols if g_row[col] == 1]
        if 'start' in g_row and pd.notna(g_row['start']):
            s, e = int(g_row['start']), int(g_row['end'])
        else:
            #Find the span inside the full text
            s = full_text.find(str(g_row['span_text']))
            e = s + len(str(g_row['span_text']))

        if s != -1:
            for t in techs:
                gold_data.append({"span": (s, e), "tech": t})

    #Run the pipeline
    predictions = run_pipeline(full_text)

    #Math for SI
    pred_spans_only = [p['span'] for p in predictions]
    gold_spans_only = [g['span'] for g in gold_data]
    si_p, si_r, si_f1 = get_si_metrics(pred_spans_only, gold_spans_only)
    all_si_scores.append(si_f1)

    #Math for full pipeline
    tc_score = 0
    if predictions and gold_data:
        match_scores = []
        for p in predictions:
            best_overlap = 0
            for g in gold_data:
                #Calculate intersection of characters
                intersect = max(0, min(p['span'][1], g['span'][1]) - max(p['span'][0], g['span'][0]))
                if intersect > 0 and p['technique'] == g['tech']:
                    #SemEval Weighting: overlap / length of predicted span
                    best_overlap = max(best_overlap, intersect / (p['span'][1] - p['span'][0]))
            match_scores.append(best_overlap)
        tc_score = sum(match_scores) / len(predictions)
    all_pipeline_scores.append(tc_score)

print("\n" + "="*40)
print(f"Mean SI F1 Score:       {np.mean(all_si_scores):.4f}")
print(f"Mean Pipeline TC F1:    {np.mean(all_pipeline_scores):.4f}")
print("="*40)

Evaluating pipeline on 357 articles...

Mean SI F1 Score:       0.2268
Mean Pipeline TC F1:    0.0771
